# FluSight: XGBoost, South Carolina
Inputs: data/CDC.csv, data/MUSC.csv, data/PRISMA.csv. Outputs stay separate by model.
Run all main cells in order. Dataset mode (default) forecasts the four weeks after the last common observation.
Only explicit Live mode checks this week's deadline and freshness for submission.
For historical checks use the launcher with -Mode Retrospective; never submit retrospective files as live forecasts.
Selection and tuning precede calibration. Forecast adjustment and quantiles use only outcomes available by each origin.
Backtests use current extracts, not historical release vintages.


## 1. Imports and Setup

In [ ]:
from pathlib import Path
import importlib.util
import subprocess
import sys
import warnings


def ensure_package(package_name: str, import_name: str | None = None) -> None:
    if importlib.util.find_spec(import_name or package_name) is None:
        raise ImportError(f"Missing {package_name}. Select the .flusight_venv interpreter or install requirements.txt.")


ensure_package("optuna")
ensure_package("statsmodels")
ensure_package("scikit-learn", "sklearn")
ensure_package("openpyxl")
ensure_package("xgboost")

import numpy as np
import optuna
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from pandas.api.types import is_numeric_dtype
from sklearn.feature_selection import mutual_info_regression
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import TimeSeriesSplit
from statsmodels.stats.outliers_influence import variance_inflation_factor
from xgboost import XGBRegressor

warnings.filterwarnings("ignore", category=RuntimeWarning)

import os
ROOT_DIR = Path(os.environ.get("FLUSIGHT_ROOT", Path.cwd())).resolve()
if not (ROOT_DIR / "flusight_config.json").is_file():
    raise ValueError("Open this notebook with its Flu folder as the working directory, or set FLUSIGHT_ROOT to that folder.")
sys.path.insert(0, str(ROOT_DIR))
from flusight_submission import (load_settings, select_reference_date, prepare_source_frames,
    check_input_dates, snapshot_inputs, temporal_evaluation, causal_adjustment,
    latest_known_ratio, export_forecast)
SETTINGS = load_settings(ROOT_DIR)
MODEL_KEY = "XGBoost"
SUBMISSION_MODE = os.environ.get("FLUSIGHT_MODE", "dataset")
if SUBMISSION_MODE not in ["dataset", "live", "retrospective"]:
    raise ValueError("FLUSIGHT_MODE must be dataset, live, or retrospective.")
CHECK_ONLY = os.environ.get("FLUSIGHT_CHECK_ONLY") == "1"
DATA_DIR = Path(os.environ.get("FLUSIGHT_DATA_DIR", str(ROOT_DIR / SETTINGS["data_folder"]))).resolve()
OUTPUT_DIR = ROOT_DIR / "outputs" / MODEL_KEY
if not CHECK_ONLY:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Using data directory: {DATA_DIR}")
print(f"Using output directory: {OUTPUT_DIR}")


## 2. Configuration

In [ ]:
SEED = 42
HORIZONS = [1, 2, 3, 4]

TARGET_SOURCE_COL = "Total.Influenza.Admissions"
original_Target_col = "Total.Influenza.Admissions_CDC"
shifted_target_column = "Target_Shifhted"
TIME_COL = "Week"

####################################################
MI_TEST_FRACTION = 0.10
TEST_WEEKS = 12
GAP_WEEKS = 1
TRAIN_START_DATE = None
TRAIN_END_DATE = None
TEST_START_DATE = None
TEST_END_DATE = None

MUST_KEEP_FEATURES = [
    "Weekly_Inpatient_Hospitalizations_MUSC",
    "Weekly_Positive_Tests_PRISMA",
]

TOP_N_FEATURES = 40
PROTECT_TOP_N = 10
VIF_THRESHOLD = 100
N_LAGS = 3
N_SPLITS = 10

RUN_OPTUNA = True
N_TRIALS = int(os.environ.get("FLUSIGHT_TRIALS", "2"))
SMALL_TARGET_THRESHOLD = 15
ADJUSTMENT_ALPHA = 0.99
FORECAST_ADJUSTMENT_ALPHA_BY_HORIZON = {
    1: ADJUSTMENT_ALPHA,
    2: ADJUSTMENT_ALPHA,
    3: ADJUSTMENT_ALPHA,
    4: ADJUSTMENT_ALPHA,
}

BASE_XGB_PARAMS = {
    "n_jobs": 1,
    "random_state": SEED,
    "objective": "reg:squarederror",
    "eval_metric": "mae",
}

DEFAULT_XGB_PARAMS = {
    "n_estimators": 500,
    "max_depth": 6,
    "learning_rate": 0.05,
    "min_child_weight": 1,
    "subsample": 0.9,
    "colsample_bytree": 0.9,
    "reg_alpha": 0.0,
    "reg_lambda": 1.0,
    **BASE_XGB_PARAMS,
}

DMA_PRIME_COLUMNS = [
    "reference_date",
    "target",
    "location_general",
    "location",
    "target_end_date",
    "value",
    "disease",
    "population",
    "training_validation",
    "estimate_projected_report",
    "imputed",
    "data_source",
    "outcome_measure",
    "output_type",
    "output_type_id",
]
DMA_LOCATION_GENERAL = "State"
DMA_LOCATION = "SC"
DMA_POPULATION = "general_population"
DMA_DATA_SOURCE = "CDC_NHSN"
DMA_OUTCOME_MEASURE_BY_DISEASE = {
    "FLU": "wk inc flu hosp",
    "RSV": "wk inc rsv hosp",
}
DMA_OUTPUT_TYPE = "quantile"
DMA_OUTPUT_TYPE_ID = 0.5


## 3. Load, Merge, and Clean Data

In [ ]:
MISSING_TOKENS = ["", " ", "NA", "N/A", "na", "null", "None", "-", "--"]


def clean_numeric_series(series: pd.Series) -> pd.Series:
    if is_numeric_dtype(series):
        return pd.to_numeric(series, errors="coerce")
    cleaned = (series.astype("string").str.strip().replace(MISSING_TOKENS, np.nan)
               .str.replace(",", "", regex=False))
    return pd.to_numeric(cleaned, errors="coerce")


def load_and_merge_data(data_dir: Path) -> pd.DataFrame:
    global SOURCE_FRAMES
    missing = [name for name in ["MUSC.csv", "PRISMA.csv", "CDC.csv"] if not (data_dir / name).is_file()]
    if missing:
        raise FileNotFoundError(f"Put {', '.join(missing)} in {data_dir} before running.")
    SOURCE_FRAMES = prepare_source_frames(
        {name: pd.read_csv(data_dir / f"{name}.csv") for name in ["MUSC", "PRISMA", "CDC"]},
        TARGET_SOURCE_COL, SUBMISSION_MODE)
    musc_1 = SOURCE_FRAMES["MUSC"].drop(columns=["State"])
    prisma_1 = SOURCE_FRAMES["PRISMA"].drop(columns=["State"])
    cdc_1 = SOURCE_FRAMES["CDC"].drop(columns=["Geographic.aggregation"]).rename(
        columns={"Week.Ending.Date": TIME_COL, TARGET_SOURCE_COL: original_Target_col})
    musc_1 = musc_1.rename(columns=lambda col: col if col == TIME_COL else f"{col}_MUSC")
    prisma_1 = prisma_1.rename(columns=lambda col: col if col == TIME_COL else f"{col}_PRISMA")
    cdc_1 = cdc_1.rename(columns=lambda col: col if col in {TIME_COL, original_Target_col} else f"{col}_CDC")
    merged = musc_1.merge(prisma_1, on=TIME_COL, how="inner", validate="one_to_one").merge(
        cdc_1, on=TIME_COL, how="inner", validate="one_to_one").sort_values(TIME_COL).reset_index(drop=True)
    for col in merged.columns:
        if col != TIME_COL:
            merged[col] = clean_numeric_series(merged[col])
    merged = merged.dropna(subset=[original_Target_col]).reset_index(drop=True)
    target = merged[original_Target_col].to_numpy(float)
    if not len(target) or not np.isfinite(target).all() or (target < 0).any() or (target != np.rint(target)).any():
        raise ValueError("CDC observed influenza admissions must be finite nonnegative integer counts.")
    return merged


df_merged_clean_no_nans = load_and_merge_data(DATA_DIR)
available_weeks = df_merged_clean_no_nans[TIME_COL].sort_values()
prediction_origin_week = pd.Timestamp(available_weeks.iloc[-1])
SUBMISSION_REFERENCE_DATE = select_reference_date(
    SUBMISSION_MODE, prediction_origin_week, os.environ.get("FLUSIGHT_REFERENCE_DATE"))
source_endings = check_input_dates(SOURCE_FRAMES, df_merged_clean_no_nans, SUBMISSION_REFERENCE_DATE, SUBMISSION_MODE)
TRAIN_START_DATE = pd.Timestamp(available_weeks.iloc[0])
TEST_END_DATE = prediction_origin_week - pd.Timedelta(weeks=1)
TEST_START_DATE = prediction_origin_week - pd.Timedelta(weeks=TEST_WEEKS)
CALIBRATION_END_DATE = TEST_START_DATE - pd.Timedelta(weeks=GAP_WEEKS + 1)
CALIBRATION_START_DATE = CALIBRATION_END_DATE - pd.Timedelta(weeks=SETTINGS["calibration_weeks"] - 1)
TRAIN_END_DATE = CALIBRATION_START_DATE - pd.Timedelta(weeks=max(HORIZONS) + GAP_WEEKS)
GAP_START_DATE = TRAIN_END_DATE + pd.Timedelta(weeks=1)
GAP_END_DATE = CALIBRATION_START_DATE - pd.Timedelta(weeks=1)
PREDICTION_ORIGIN_WEEK = prediction_origin_week
print("Merged shape:", df_merged_clean_no_nans.shape)
print("Source endings:", source_endings)
print("Reference date:", SUBMISSION_REFERENCE_DATE.date(), "Mode:", SUBMISSION_MODE)
print(f"Selection/tuning labels: {TRAIN_START_DATE.date()} through {TRAIN_END_DATE.date()}")
print(f"Calibration targets: {CALIBRATION_START_DATE.date()} through {CALIBRATION_END_DATE.date()}")
print(f"Test targets: {TEST_START_DATE.date()} through {TEST_END_DATE.date()}")
print("Forecast origin:", prediction_origin_week.date())
print("Prediction weeks:", [str((prediction_origin_week + pd.Timedelta(weeks=h)).date()) for h in HORIZONS])
INPUT_SNAPSHOT = {} if CHECK_ONLY else snapshot_inputs(ROOT_DIR, DATA_DIR, MODEL_KEY, SUBMISSION_REFERENCE_DATE, SUBMISSION_MODE)
display(df_merged_clean_no_nans[[TIME_COL, original_Target_col]].tail())


## 4. Shared Metrics and Feature Utilities

In [ ]:
def percent_agreement_fraction(y_true, y_pred, small_threshold=SMALL_TARGET_THRESHOLD):
    y_true = np.asarray(y_true, dtype=float).ravel()
    y_pred = np.asarray(y_pred, dtype=float).ravel()

    ignore = y_true < small_threshold
    use = ~ignore

    yt = y_true[use]
    yp = y_pred[use]

    used = int(yt.size)
    ignored = int(ignore.sum())

    if used == 0:
        return {"score": np.nan, "used": used, "ignored": ignored}

    denom = np.maximum(yt, yp)
    score = np.divide(np.minimum(yt, yp), denom, out=np.zeros_like(denom), where=denom != 0)
    return {"score": float(np.mean(score)), "used": used, "ignored": ignored}


def safe_pct_change(curr: pd.Series, prev: pd.Series) -> pd.Series:
    out = (curr - prev) / prev
    out = out.where(prev.notna(), np.nan)
    out = out.mask(prev == 0, 0.0)
    return out


def safe_ratio(curr: pd.Series, prev: pd.Series) -> pd.Series:
    out = curr / prev
    out = out.where(prev.notna(), np.nan)
    out = out.mask(prev == 0, 1.0)
    return out


def get_vif(df_: pd.DataFrame) -> pd.DataFrame:
    if df_.shape[1] == 0:
        return pd.DataFrame(columns=["Feature", "VIF"])
    if df_.shape[1] == 1:
        return pd.DataFrame({"Feature": df_.columns, "VIF": [np.nan]})
    values = df_.astype(float).replace([np.inf, -np.inf], np.nan)
    values = values.fillna(values.median()).fillna(0.0)
    # Correlated clinical indicators can emit the same rank warning hundreds of times.
    # Preserve the computation and report that diagnostic once rather than flooding the run.
    with warnings.catch_warnings(record=True) as diagnostics:
        warnings.simplefilter("always")
        vifs = [variance_inflation_factor(values.values, i) for i in range(values.shape[1])]
    if diagnostics and not globals().get("VIF_DIAGNOSTIC_SHOWN", False):
        print("VIF diagnostic: correlated/rank-deficient predictors detected; unprotected high-VIF predictors are pruned.")
        globals()["VIF_DIAGNOSTIC_SHOWN"] = True
    return pd.DataFrame({"Feature": values.columns, "VIF": vifs}).sort_values(
        "VIF", ascending=False).reset_index(drop=True)


def add_adjusted_predictions(pa_table: pd.DataFrame, alpha: float = ADJUSTMENT_ALPHA) -> pd.DataFrame:
    return causal_adjustment(pa_table, alpha=alpha)


def get_last_current_ratio(adjusted_pa_table: pd.DataFrame) -> float:
    ratios = pd.to_numeric(adjusted_pa_table["current_ratio"], errors="coerce")
    ratios = ratios.replace([np.inf, -np.inf], np.nan).dropna()

    if ratios.empty:
        return 1.0

    return float(ratios.iloc[-1])


def adjust_final_forecast(raw_forecast: float, last_current_ratio: float, alpha: float) -> float:
    return float(max(raw_forecast * last_current_ratio * alpha, 0))


## 5. Reusable Horizon Pipeline

In [ ]:
def create_shifted_target(df: pd.DataFrame, horizon: int) -> tuple[pd.DataFrame, str]:
    horizon_shifted_target_col = f"{original_Target_col}_T+{horizon}"
    df_h = df.copy()
    df_h[TIME_COL] = pd.to_datetime(df_h[TIME_COL], errors="coerce")
    df_h = df_h.dropna(subset=[TIME_COL]).sort_values(TIME_COL).reset_index(drop=True)
    df_h[horizon_shifted_target_col] = df_h[original_Target_col].shift(-horizon)
    return df_h, horizon_shifted_target_col


def select_features_mi_vif(df_h: pd.DataFrame, horizon_shifted_target_col: str, horizon: int) -> dict:
    df_mi_full = df_h.copy().sort_values(TIME_COL).reset_index(drop=True)

    if horizon_shifted_target_col not in df_mi_full.columns:
        raise KeyError(f"{horizon_shifted_target_col} is missing.")

    df_mi = df_mi_full.dropna(subset=[horizon_shifted_target_col]).reset_index(drop=True)

    label_weeks = df_mi[TIME_COL] + pd.Timedelta(weeks=horizon)
    train_mi = df_mi.loc[label_weeks.between(TRAIN_START_DATE, TRAIN_END_DATE)].copy()
    gap_mi = df_mi.loc[label_weeks.between(GAP_START_DATE, GAP_END_DATE)].copy()
    test_mi = df_mi.loc[label_weeks >= CALIBRATION_START_DATE].copy()
    if len(train_mi) < 80:
        raise ValueError("At least 80 initial rows are required before calibration for selection and tuning.")

    feature_candidate_cols = [
        col for col in train_mi.columns
        if col not in [TIME_COL, horizon_shifted_target_col]
    ]

    cols_with_train_nans = [col for col in feature_candidate_cols if train_mi[col].isna().any()]
    kept_feature_cols = [col for col in feature_candidate_cols if col not in cols_with_train_nans]

    train_numeric = train_mi[kept_feature_cols + [horizon_shifted_target_col]].select_dtypes(include=[np.number]).copy()
    train_numeric = train_numeric.dropna(axis=1, how="all")
    train_numeric = train_numeric.dropna(subset=[horizon_shifted_target_col]).reset_index(drop=True)

    X_train_mi = train_numeric.drop(columns=[horizon_shifted_target_col])
    y_train_mi = train_numeric[horizon_shifted_target_col]
    X_train_mi = X_train_mi.replace([np.inf, -np.inf], np.nan)
    X_train_mi = X_train_mi.fillna(X_train_mi.median()).fillna(0.0)

    mi_scores = mutual_info_regression(X_train_mi, y_train_mi, random_state=SEED)
    mi_df = (
        pd.DataFrame({"Feature": X_train_mi.columns, "MI_Score": mi_scores})
        .sort_values("MI_Score", ascending=False)
        .reset_index(drop=True)
    )

    must_keep_available = [col for col in MUST_KEEP_FEATURES if col in X_train_mi.columns]
    selected_features = mi_df["Feature"].head(TOP_N_FEATURES).tolist()
    selected_features = list(dict.fromkeys(selected_features + must_keep_available))

    protected_features = set(mi_df["Feature"].head(PROTECT_TOP_N).tolist() + must_keep_available)
    mi_lookup = dict(zip(mi_df["Feature"], mi_df["MI_Score"]))

    df_vif = X_train_mi[selected_features].copy()
    constant_cols = [
        col for col in df_vif.columns
        if df_vif[col].nunique(dropna=False) <= 1 and col not in protected_features
    ]
    df_vif = df_vif.drop(columns=constant_cols)

    dropped_features = []
    while df_vif.shape[1] > 1:
        vif_df = get_vif(df_vif)
        candidates = vif_df[
            (vif_df["VIF"] > VIF_THRESHOLD)
            & (~vif_df["Feature"].isin(protected_features))
        ].copy()
        if candidates.empty:
            break

        candidates["MI_Score"] = candidates["Feature"].map(mi_lookup)
        feature_to_drop = candidates.sort_values(["MI_Score", "VIF"], ascending=[True, False]).iloc[0]["Feature"]
        dropped_features.append(feature_to_drop)
        df_vif = df_vif.drop(columns=[feature_to_drop])

    vif_df = get_vif(df_vif)
    final_features = df_vif.columns.tolist()

    return {
        "df_mi_full": df_mi_full,
        "mi_df": mi_df,
        "vif_df": vif_df,
        "final_features": final_features,
        "dropped_features": dropped_features,
        "cols_with_train_nans": cols_with_train_nans,
        "mi_split_sizes": {
            "train": len(train_mi),
            "gap": len(gap_mi),
            "test": len(test_mi),
        },
    }


def build_selected_frame(df_mi_full: pd.DataFrame, selected_features: list[str], horizon_shifted_target_col: str) -> pd.DataFrame:
    final_columns = [TIME_COL] + selected_features + [original_Target_col, horizon_shifted_target_col]
    final_columns = list(dict.fromkeys(final_columns))
    return df_mi_full[final_columns].copy()


def engineer_features(vif_df_before_feature_engineering: pd.DataFrame, horizon_shifted_target_col: str, horizon: int) -> dict:
    df = vif_df_before_feature_engineering.copy()
    df[TIME_COL] = pd.to_datetime(df[TIME_COL], errors="coerce")
    df = df.dropna(subset=[TIME_COL]).sort_values(TIME_COL).reset_index(drop=True)

    expected_label = df[original_Target_col].shift(-horizon)
    label_ok = np.isclose(
        df[horizon_shifted_target_col].to_numpy(dtype=float),
        expected_label.to_numpy(dtype=float),
        equal_nan=True,
    ).all()
    if not label_ok:
        raise ValueError(f"{horizon_shifted_target_col} is not aligned with {original_Target_col}.shift(-{horizon}).")

    df["week_num"] = df[TIME_COL].dt.isocalendar().week.astype(int)
    df["sin_week"] = np.sin(2 * np.pi * df["week_num"] / 52)
    df["cos_week"] = np.cos(2 * np.pi * df["week_num"] / 52)

    season_cols = ["sin_week", "cos_week"]
    raw_feature_cols = [
        col for col in df.columns
        if col not in [TIME_COL, original_Target_col, horizon_shifted_target_col, "week_num"] + season_cols
    ]

    new_features = {}

    for col in raw_feature_cols:
        s = df[col]
        for lag in range(1, N_LAGS + 1):
            new_features[f"{col}_lag{lag}"] = s.shift(lag)

        new_features[f"{col}_pct_change1"] = safe_pct_change(s, s.shift(1))
        new_features[f"{col}_pct_change2"] = safe_pct_change(s.shift(1), s.shift(2))
        new_features[f"{col}_ratio1"] = safe_ratio(s, s.shift(1))
        new_features[f"{col}_ratio2"] = safe_ratio(s.shift(1), s.shift(2))
        new_features[f"{col}_rollmean2"] = s.rolling(2).mean()
        new_features[f"{col}_rollmean4"] = s.rolling(4).mean()
        new_features[f"{col}_speed"] = s.diff()
        new_features[f"{col}_acc"] = s.diff().diff()

    y_hist = df[original_Target_col]
    new_features[original_Target_col] = y_hist

    for lag in range(1, N_LAGS + 1):
        new_features[f"{original_Target_col}_lag{lag}"] = y_hist.shift(lag)

    new_features[f"{original_Target_col}_pct_change1"] = safe_pct_change(y_hist, y_hist.shift(1))
    new_features[f"{original_Target_col}_pct_change2"] = safe_pct_change(y_hist.shift(1), y_hist.shift(2))
    new_features[f"{original_Target_col}_ratio1"] = safe_ratio(y_hist, y_hist.shift(1))
    new_features[f"{original_Target_col}_ratio2"] = safe_ratio(y_hist.shift(1), y_hist.shift(2))
    new_features[f"{original_Target_col}_rollmean2"] = y_hist.rolling(2).mean()
    new_features[f"{original_Target_col}_rollmean4"] = y_hist.rolling(4).mean()
    new_features[f"{original_Target_col}_speed"] = y_hist.diff()
    new_features[f"{original_Target_col}_acc"] = y_hist.diff().diff()

    features_df = pd.DataFrame(new_features, index=df.index)

    df_full = pd.concat(
        [df[[TIME_COL] + raw_feature_cols + season_cols], features_df],
        axis=1,
    ).replace([np.inf, -np.inf], np.nan)

    df_full[shifted_target_column] = df[horizon_shifted_target_col]

    model_feature_cols = [col for col in df_full.columns if col not in [TIME_COL, shifted_target_column]]

    vif_df_after_feature_engineering = df_full.dropna(
        subset=model_feature_cols + [shifted_target_column]
    ).reset_index(drop=True)

    df_for_forecast = df_full.dropna(subset=model_feature_cols).tail(1).reset_index(drop=True)
    if df_for_forecast.empty or pd.Timestamp(df_for_forecast[TIME_COL].iloc[0]) != prediction_origin_week:
        raise ValueError("Complete predictors are unavailable for the latest week. Fill missing source predictors before forecasting.")

    return {
        "before": df,
        "full": df_full,
        "training": vif_df_after_feature_engineering,
        "forecast": df_for_forecast,
        "model_feature_cols": model_feature_cols,
    }


## 6. Reusable Model Training and Forecasting

In [ ]:
def tune_xgb_params(X: pd.DataFrame, y: pd.Series, horizon: int) -> tuple[dict, float]:
    if not RUN_OPTUNA:
        return DEFAULT_XGB_PARAMS.copy(), np.nan

    n_splits = min(N_SPLITS, max(2, len(X) // 20))
    tscv = TimeSeriesSplit(n_splits=n_splits, gap=horizon)

    def objective(trial):
        params = {
            "n_estimators": trial.suggest_int("n_estimators", 200, 800),
            "max_depth": trial.suggest_int("max_depth", 2, 12),
            "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.30, log=True),
            "min_child_weight": trial.suggest_int("min_child_weight", 1, 10),
            "subsample": trial.suggest_float("subsample", 0.6, 1.0),
            "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
            "reg_alpha": trial.suggest_float("reg_alpha", 0.0, 1.0),
            "reg_lambda": trial.suggest_float("reg_lambda", 0.1, 10.0, log=True),
            **BASE_XGB_PARAMS,
        }
        oof_preds = np.full(len(X), np.nan, dtype=float)
        y_true = y.to_numpy(dtype=float)
        for train_idx, val_idx in tscv.split(X):
            model = XGBRegressor(**params)
            model.fit(X.iloc[train_idx], y.iloc[train_idx], verbose=False)
            oof_preds[val_idx] = np.maximum(model.predict(X.iloc[val_idx]), 1)
        valid_mask = ~np.isnan(oof_preds)
        return percent_agreement_fraction(y_true[valid_mask], oof_preds[valid_mask], SMALL_TARGET_THRESHOLD)["score"]

    sampler = optuna.samplers.TPESampler(seed=SEED)
    study = optuna.create_study(direction="maximize", sampler=sampler)
    study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=False)
    best_params = study.best_params.copy()
    best_params.update(BASE_XGB_PARAMS)
    return best_params, float(study.best_value)


def fit_predict_holdout(modeling_df: pd.DataFrame, horizon: int) -> dict:
    feature_cols = [col for col in modeling_df.columns if col not in [TIME_COL, shifted_target_column]]
    return temporal_evaluation(
        modeling_df, feature_cols, horizon, lambda params: XGBRegressor(**params), tune_xgb_params,
        TRAIN_START_DATE, TRAIN_END_DATE, CALIBRATION_START_DATE, TEST_START_DATE, TEST_END_DATE,
        prediction_origin_week, shifted_target_column, ADJUSTMENT_ALPHA, SETTINGS["calibration_min_rows"])


def fit_final_forecast(modeling_df: pd.DataFrame, forecast_df: pd.DataFrame, feature_cols: list[str], best_params: dict, horizon: int) -> dict:
    final_model_all = XGBRegressor(**best_params)
    final_model_all.fit(modeling_df[feature_cols], modeling_df[shifted_target_column], verbose=False)
    forecast_origin_week = pd.Timestamp(forecast_df[TIME_COL].iloc[0])
    forecast_week = forecast_origin_week + pd.Timedelta(days=7 * horizon)
    forecast_value = float(np.maximum(final_model_all.predict(forecast_df[feature_cols])[0], 0))
    return {"forecast_origin_week": forecast_origin_week, "forecast_week": forecast_week, "forecast": forecast_value, "forecast_rounded": int(round(forecast_value)), "model": final_model_all}


def run_horizon(df: pd.DataFrame, horizon: int) -> dict:
    print(f"\n===== Running T+{horizon} =====")
    df_h, horizon_shifted_target_col = create_shifted_target(df, horizon)
    feature_selection = select_features_mi_vif(df_h, horizon_shifted_target_col, horizon)
    selected_frame = build_selected_frame(feature_selection["df_mi_full"], feature_selection["final_features"], horizon_shifted_target_col)
    engineered = engineer_features(selected_frame, horizon_shifted_target_col, horizon)
    holdout = fit_predict_holdout(engineered["training"], horizon)
    forecast = fit_final_forecast(engineered["training"], engineered["forecast"], holdout["feature_cols"], holdout["best_params"], horizon)
    summary = {"horizon": f"T+{horizon}", "horizon_weeks": horizon, "target": original_Target_col, "latest_observed_week": pd.Timestamp(df[TIME_COL].iloc[-1]).date().isoformat(), "latest_observed_value": float(df[original_Target_col].iloc[-1]), "forecast_origin_week": forecast["forecast_origin_week"].date().isoformat(), "forecast_week": forecast["forecast_week"].date().isoformat(), "forecast": forecast["forecast"], "forecast_rounded": forecast["forecast_rounded"], "mi_train_rows": feature_selection["mi_split_sizes"]["train"], "mi_gap_rows": feature_selection["mi_split_sizes"]["gap"], "mi_test_rows": feature_selection["mi_split_sizes"]["test"], "selected_features_before_engineering": len(feature_selection["final_features"]), "model_features_after_engineering": len(holdout["feature_cols"]), "holdout_train_rows": len(holdout["train_df"]), "holdout_gap_rows": len(holdout["gap_df"]), "holdout_test_rows": len(holdout["test_df"]), "holdout_mae": holdout["test_mae"], "holdout_percent_agreement": holdout["test_pa"]["score"], "holdout_pa_used": holdout["test_pa"]["used"], "holdout_pa_ignored": holdout["test_pa"]["ignored"], "adjusted_holdout_mae": holdout["adjusted_test_mae"], "adjusted_holdout_percent_agreement": holdout["adjusted_test_pa"]["score"], "adjusted_holdout_pa_used": holdout["adjusted_test_pa"]["used"], "adjusted_holdout_pa_ignored": holdout["adjusted_test_pa"]["ignored"], "best_cv_percent_agreement": holdout["best_cv_pa"]}
    return {"summary": summary, "feature_selection": feature_selection, "selected_frame": selected_frame, "engineered": engineered, "holdout": holdout, "forecast": forecast}


In [ ]:
def next_saturday(run_date: pd.Timestamp | None = None) -> pd.Timestamp:
    current_date = pd.Timestamp.today().normalize() if run_date is None else pd.Timestamp(run_date).normalize()
    days_until_saturday = (5 - current_date.weekday()) % 7
    if days_until_saturday == 0:
        days_until_saturday = 7
    return current_date + pd.Timedelta(days=days_until_saturday)


def format_mdy(value) -> str:
    value = pd.Timestamp(value)
    return f"{value.month}/{value.day}/{value.year}"


def infer_dma_disease_name() -> str:
    disease_context = " ".join([
        str(TARGET_SOURCE_COL),
        str(original_Target_col),
        str(Path.cwd().name),
    ]).upper()

    if "FLU" in disease_context:
        return "FLU"
    if "INFLUENZA" in disease_context or "FLU" in disease_context:
        return "FLU"
    if "RSV" in disease_context:
        return "RSV"

    raise ValueError(f"Could not infer DMA disease from: {disease_context}")


def make_dma_prime_rows(
    frame: pd.DataFrame,
    values: np.ndarray,
    horizon: int,
    training_validation: int,
    reference_date_value: str,
    disease: str,
    outcome_measure: str,
) -> pd.DataFrame:
    target_weeks = pd.to_datetime(frame[TIME_COL]) + pd.to_timedelta(7 * horizon, unit="D")
    return pd.DataFrame({
        "reference_date": reference_date_value,
        "target": "",
        "location_general": DMA_LOCATION_GENERAL,
        "location": DMA_LOCATION,
        "target_end_date": [format_mdy(value) for value in target_weeks],
        "value": np.asarray(values, dtype=float),
        "disease": disease,
        "population": DMA_POPULATION,
        "training_validation": training_validation,
        "estimate_projected_report": horizon - 1,
        "imputed": 0,
        "data_source": DMA_DATA_SOURCE,
        "outcome_measure": outcome_measure,
        "output_type": DMA_OUTPUT_TYPE,
        "output_type_id": DMA_OUTPUT_TYPE_ID,
    })[DMA_PRIME_COLUMNS]


def build_dma_prime_evaluation_table(
    horizon_outputs: dict[int, dict],
    reference_date: pd.Timestamp | None = None,
) -> pd.DataFrame:
    reference_date_value = format_mdy(SUBMISSION_REFERENCE_DATE if reference_date is None else reference_date)
    disease = infer_dma_disease_name()
    outcome_measure = DMA_OUTCOME_MEASURE_BY_DISEASE[disease]
    rows = []

    for horizon in HORIZONS:
        result = horizon_outputs[horizon]
        holdout = result["holdout"]
        feature_cols = holdout["feature_cols"]

        train_df = holdout["train_df"].copy().sort_values(TIME_COL).reset_index(drop=True)
        test_df = holdout["test_df"].copy().sort_values(TIME_COL).reset_index(drop=True)

        holdout_model = XGBRegressor(**holdout["best_params"])
        holdout_model.fit(train_df[feature_cols], train_df[shifted_target_column])

        train_values = np.maximum(holdout_model.predict(train_df[feature_cols]), 1)
        test_values = holdout["adjusted_comparison_test"]["adjusted_predicted"].to_numpy(dtype=float)

        rows.append(
            make_dma_prime_rows(
                frame=train_df,
                values=train_values,
                horizon=horizon,
                training_validation=1,
                reference_date_value=reference_date_value,
                disease=disease,
                outcome_measure=outcome_measure,
            )
        )
        rows.append(
            make_dma_prime_rows(
                frame=test_df,
                values=test_values,
                horizon=horizon,
                training_validation=0,
                reference_date_value=reference_date_value,
                disease=disease,
                outcome_measure=outcome_measure,
            )
        )

    return pd.concat(rows, ignore_index=True)


## 7. Run All Four Forecast Horizons

In [ ]:
actual_cdc = SOURCE_FRAMES["CDC"].copy()
actual_cdc[TIME_COL] = pd.to_datetime(actual_cdc["Week.Ending.Date"], errors="coerce")
actual_cdc[original_Target_col] = clean_numeric_series(actual_cdc[TARGET_SOURCE_COL])
actual_period_mask = actual_cdc[TIME_COL].between(TRAIN_START_DATE, prediction_origin_week)
actual_admissions_csv = (
    actual_cdc.loc[actual_period_mask, [TIME_COL, "Geographic.aggregation", original_Target_col]]
    .rename(columns={"Geographic.aggregation": "State", original_Target_col: "Total_FLU_Admissions"})
    .sort_values(TIME_COL)
)
actual_admissions_csv[TIME_COL] = actual_admissions_csv[TIME_COL].map(format_mdy)
actual_admissions_path = OUTPUT_DIR / "FLU_actual_admissions_train_test.csv"
actual_admissions_csv.to_csv(actual_admissions_path, index=False)
print(f"Saved actual admissions CSV to: {actual_admissions_path}")

horizon_outputs = {}
summary_rows = []

for horizon in HORIZONS:
    horizon_result = run_horizon(df_merged_clean_no_nans, horizon)
    horizon_outputs[horizon] = horizon_result
    summary_rows.append(horizon_result["summary"])

forecast_summary = pd.DataFrame(summary_rows)
summary_path = OUTPUT_DIR / "combined_flu_forecast_t1_t4_summary.csv"
forecast_summary.to_csv(summary_path, index=False)

for horizon, result in horizon_outputs.items():
    result["holdout"]["calibration_history"].to_csv(
        OUTPUT_DIR / f"causal_calibration_history_T_plus_{horizon}.csv", index=False)
    result["holdout"]["quantile_metrics"].to_csv(
        OUTPUT_DIR / f"quantile_holdout_metrics_T_plus_{horizon}.csv", index=False)
    result["holdout"]["comparison_test"].to_csv(
        OUTPUT_DIR / f"combined_flu_PA_table_T_plus_{horizon}.csv",
        index=False,
    )
    result["holdout"]["adjusted_comparison_test"].to_csv(
        OUTPUT_DIR / f"combined_flu_adjusted_PA_table_T_plus_{horizon}.csv",
        index=False,
    )

selected_feature_rows = []
for horizon, result in horizon_outputs.items():
    for rank, feature in enumerate(result["feature_selection"]["final_features"], start=1):
        selected_feature_rows.append({"horizon": f"T+{horizon}", "rank": rank, "feature": feature})

selected_features_df = pd.DataFrame(selected_feature_rows)
selected_features_path = OUTPUT_DIR / "combined_flu_selected_features_t1_t4.csv"
selected_features_df.to_csv(selected_features_path, index=False)


dma_prime_evaluation = build_dma_prime_evaluation_table(horizon_outputs)
dma_prime_csv_path = OUTPUT_DIR / "DMA-PRIME_Model_Evaluation_FLU.csv"
dma_prime_excel_path = OUTPUT_DIR / "DMA-PRIME_Model_Evaluation_FLU.xlsx"
dma_prime_evaluation.to_csv(dma_prime_csv_path, index=False)
dma_prime_evaluation.to_excel(dma_prime_excel_path, index=False)

print(f"Saved forecast summary to: {summary_path}")
print(f"Saved selected features to: {selected_features_path}")
print(f"Saved DMA-PRIME CSV to: {dma_prime_csv_path}")
print(f"Saved DMA-PRIME Excel to: {dma_prime_excel_path}")
display(forecast_summary)
display(dma_prime_evaluation.head())

for horizon, result in horizon_outputs.items():
    print(f"T+{horizon} PA table")
    display(result["holdout"]["comparison_test"])
    print(f"T+{horizon} adjusted PA table")
    display(result["holdout"]["adjusted_comparison_test"])

display(
    forecast_summary[[
        "horizon",
        "forecast_origin_week",
        "forecast_week",
        "forecast_rounded",
        "holdout_mae",
        "holdout_percent_agreement",
        "adjusted_holdout_mae",
        "adjusted_holdout_percent_agreement",
    ]]
)


## 8. Hold-out Diagnostic Plots

In [ ]:
for horizon, result in horizon_outputs.items():
    comparison = result["holdout"]["adjusted_comparison_test"]

    plt.figure(figsize=(12, 5))
    plt.plot(comparison["Target_Week"], comparison["Actual"], marker="o", label="Actual")
    plt.plot(
        comparison["Target_Week"],
        comparison["adjusted_predicted"],
        marker="s",
        color="green",
        label="Adjusted Predicted",
    )

    for x, y in zip(comparison["Target_Week"], comparison["Actual"]):
        plt.annotate(f"{y:.1f}", (x, y), textcoords="offset points", xytext=(0, 8), ha="center", fontsize=8)

    for x, y in zip(comparison["Target_Week"], comparison["adjusted_predicted"]):
        plt.annotate(f"{y:.1f}", (x, y), textcoords="offset points", xytext=(0, -12), ha="center", fontsize=8, color="green")

    plt.title(f"T+{horizon} Adjusted PA: Actual vs Adjusted Predicted")
    plt.xlabel("Target_Week")
    plt.ylabel(original_Target_col)
    plt.xticks(rotation=45)
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

## 9. Forecast Summary Plot

In [ ]:
plot_df = forecast_summary.copy()
plot_df["forecast_week"] = pd.to_datetime(plot_df["forecast_week"])

plt.figure(figsize=(10, 5))
plt.plot(plot_df["forecast_week"], plot_df["forecast"], marker="o", label="Forecast")

for x, y, label in zip(plot_df["forecast_week"], plot_df["forecast"], plot_df["horizon"]):
    plt.annotate(f"{label}: {y:.1f}", (x, y), textcoords="offset points", xytext=(0, 8), ha="center", fontsize=9)

plt.title("FLU Admission Forecasts by Horizon")
plt.xlabel("Forecast Week")
plt.ylabel(original_Target_col)
plt.xticks(rotation=45)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 10. Inspect Per-Horizon Artifacts

In [ ]:
#print("VIF DataFrame for T+1:")
#horizon_outputs[1]["feature_selection"]["vif_df"]


In [ ]:
#print("VIF DataFrame for T+2:")
#horizon_outputs[2]["feature_selection"]["vif_df"]

In [ ]:
#print("VIF DataFrame for T+3:")
#horizon_outputs[3]["feature_selection"]["vif_df"]

In [ ]:
#print("VIF DataFrame for T+4:")
#horizon_outputs[4]["feature_selection"]["vif_df"]

# Final Forecast Models


## 1. Train Final Models And Forecast


In [ ]:
final_forecast_outputs = {}
final_forecast_rows = []

for horizon in HORIZONS:
    result = horizon_outputs[horizon]

    engineered = result["engineered"]
    holdout = result["holdout"]

    final_forecast = fit_final_forecast(
        modeling_df=engineered["training"],
        forecast_df=engineered["forecast"],
        feature_cols=holdout["feature_cols"],
        best_params=holdout["best_params"],
        horizon=horizon,
    )

    final_forecast_outputs[horizon] = final_forecast

    final_forecast_rows.append({
        "horizon": f"T+{horizon}",
        "Origin Week": final_forecast["forecast_origin_week"],
        "Target Week": final_forecast["forecast_week"],
        "Forecast": final_forecast["forecast"],
        "Forecast_Rounded": final_forecast["forecast_rounded"],
        "training_rows": len(engineered["training"]),
        "feature_count": len(holdout["feature_cols"]),
    })


## 2. Display Final Forecast Table


In [ ]:
# Freeze the same adjustment used for chronological calibration; no post-test alpha edits.
final_points = {}
for horizon in HORIZONS:
    result = horizon_outputs[horizon]
    X_next = result["engineered"]["forecast"][result["holdout"]["feature_cols"]]
    ratio = latest_known_ratio(result["holdout"]["calibration_history"], prediction_origin_week)
    alpha = FORECAST_ADJUSTMENT_ALPHA_BY_HORIZON[horizon]
    raw = float(final_forecast_outputs[horizon]["forecast"])
    final_points[horizon] = adjust_final_forecast(raw, ratio, alpha)
    globals()[f"X_next_{horizon}"] = X_next
    globals()[f"last_current_ratio_T{horizon}"] = ratio
    globals()[f"Alpha_T{horizon}"] = alpha
    globals()[f"next_forecast_{horizon}_raw"] = raw
    globals()[f"next_forecast_{horizon}"] = final_points[horizon]
    print(f"T+{horizon}: raw={raw:.1f}, adjusted={final_points[horizon]:.1f}, known ratio={ratio:.3f}")


## 3. Quantile Forecast Table


In [ ]:
# Historical forecast errors, not the standard deviation of two predicted counts.
quantile_forecast_df = export_forecast(
    ROOT_DIR, SETTINGS, MODEL_KEY, SUBMISSION_REFERENCE_DATE, prediction_origin_week,
    horizon_outputs, final_points, SUBMISSION_MODE)
display(quantile_forecast_df)


## 4. Access Final Trained Models


In [ ]:
final_model_T1 = final_forecast_outputs[1]["model"]
final_model_T2 = final_forecast_outputs[2]["model"]
final_model_T3 = final_forecast_outputs[3]["model"]
final_model_T4 = final_forecast_outputs[4]["model"]